In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.plot  # submodule is not imported by `import rasterio` alone
import torch
from IPython.display import Image  # For displaying the training GIF

# Use this to load the functions used in this notebook
# Call functions as deepssf.function_name
import deepssf

print(f"deepssf {deepssf.__version__}")
print(f"torch {torch.__version__}")

# Setup the script

## Set random seed and the device for training the model

If you have a GPU, this is where it will be specified (automatically).

In [ ]:
# Reproducibility — fix random seeds so results are the same on every run.
# These matter here: make_dataloaders() uses random_split, so without a seed the
# train/val/test partition changes every run and two training configurations
# cannot be compared.
torch.manual_seed(42)
np.random.seed(42)

# DEVICE is capitalised — module-level constant (set once, never reassigned).
# get_device() returns 'cuda' if a GPU is available, 'mps' on Apple Silicon,
# or 'cpu' otherwise.  All tensors and the model are moved to this device.
DEVICE = deepssf.get_device()
print(f"Device: {DEVICE}")


## Set script paths
If you have cloned/forked/downloaded the GitHub repo (https://github.com/swforrest/deepSSF_package) you shouldn't need to change these, but for your own analyses you will want to set these to point to where your data is and where you want outputs saved.

*All-caps names below are module-level constants — set once here and referenced throughout the notebook without being reassigned.  Python has no built-in constant keyword, so capitalisation is the community convention (PEP 8) to signal "do not change this value".*

In [ ]:
DATA_DIR = Path("data")  # location of the bundled sample data
MAPPING_DIR = Path("mapping")  # location of the landcover mapping files
CSV_PATH = DATA_DIR / "pigs_east_alligator_steps_covs_20261015.csv"  # GPS tracking data

# NOTE: 's2' and 's2_dir' are RESERVED keys in load_environmental_layers().
# 's2_dir' triggers the monthly-composite loader (load_s2_data), and a plain
# 's2' entry is skipped by the per-file loop — so a single multi-band GeoTIFF
# must be given any *other* name, otherwise nothing is loaded and the function
# raises "raster_transform could not be set".
LAYER_PATHS = {
    # Topographic layers (DEM, slope)
    "dem": str(MAPPING_DIR / "DEM_3112_crop.tif"),
    "slope": str(MAPPING_DIR / "slope_3112_crop.tif"),
    "water": str(MAPPING_DIR / "water_layer_east_alligator.tif"),

    # 3-band Sentinel-2 composite: NDVI, NDWI, NDMI (20 m, EPSG:3112)
    # "s2_indices": str(MAPPING_DIR / "feral_pig_aoi_S2_indices_2025-09-12_to_2026-06-23.tif"),

    # Alternative — all 12 S2 bands (12 spatial channels instead of 3, slower to train):
    # "s2_median": str(MAPPING_DIR / "feral_pig_aoi_S2_median_full_2025-09-12_to_2026-06-23.tif"),

    # AlphaEarth embeddings (2025)
    "alphaearth": str(MAPPING_DIR / "feral_pig_aoi_alphaearth_2025_crop.tif"),
}
                 
OUTPUT_DIR = deepssf.get_output_dir(base="outputs/alphaearth")   # figures and model checkpoints go here - function adds date and index
print(f"Output directory: {OUTPUT_DIR}")
SNAPSHOT_DIR = OUTPUT_DIR / "training_snapshots"  # per-batch prediction-map snapshots

# Import data
The csv file is imported with a timestamp and an x and y coordinate. In our case the data was already in a projected format that is specific to Australia (EPSG:3112), which is the same crs as the raster data which will be imported.

In [ ]:
raw_df = pd.read_csv(CSV_PATH)
print(f"Raw GPS fixes : {len(raw_df):,}")
print(f"Individuals   : {raw_df['name'].nunique()}")

# Sort by the individual ID and timestamp to ensure correct temporal order
raw_df = raw_df.sort_values(by=["name", "t1_"]).reset_index(drop=True)

# Convert the 'time' column to the correct timezone (here Australia/Darwin), then
# drop the offset.  prepare_movement_df() re-parses timestamps with utc=True, so a
# tz-aware Darwin column would be converted straight back to UTC and hour_t1 would
# be 9.5 h out — making the diel covariates wrong.  Localising to Darwin and then
# going tz-naive keeps the local clock time that the sin/cos encodings need.
raw_df["t1_"] = (
        pd.to_datetime(raw_df["t1_"], utc=True)
        .dt.tz_convert("Australia/Darwin")
        .dt.tz_localize(None)
    )

raw_df.head()

## `load_environmental_layers()`

Reads each GeoTIFF listed in LAYER_PATHS, min–max

- Scales every band to [0, 1], and returns:
    * `env_layers`       — dict mapping layer name → NumPy array: 2-D (H × W) for single-band GeoTIFFs, 3-D (bands × H × W) for multi-band ones, where each band becomes its own model input channel
    * `raster_transform` — Affine object that converts pixel (row, col) indices to the projected coordinates (easting, northing) of the raster(s), which should be in the same CRS as the GPS data.  This transform is used whenever we need to translate between pixel space and real-world coordinates.

**Two things to watch for:**

1. The keys `'s2'` and `'s2_dir'` are *reserved* for the monthly Sentinel-2 composite loader — a single GeoTIFF passed under the name `'s2'` is silently skipped (see the note in the paths cell above).
2. The rescaling uses a **single global min/max across all bands** of a file, and computes `(data - lo) / hi` rather than `(data - lo) / (hi - lo)`. Our index raster is `int16` scaled by 10 000 with ±10 000 fill values, so the loaded values span [0, 2] rather than [0, 1] and the three indices are not individually normalised. That is workable (the CNN can rescale internally) but if it hurts training, pre-scale each band yourself and save as `.npy` — non-`.tif` paths in `LAYER_PATHS` are loaded with `np.load(..., mmap_mode='r')` and left untouched, though at least one real `.tif` must remain in the dict so the transform can be set.


In [ ]:
env_layers, raster_transform = deepssf.load_environmental_layers(LAYER_PATHS)
print("Layers loaded:", list(env_layers.keys()))
for name, arr in env_layers.items():
    print(f"  {name}: shape={arr.shape}, min={arr.min():.3f}, max={arr.max():.3f}")

# Each band of a multi-band layer becomes one CNN input channel, so count bands
# (not layers) here and reuse the total when setting ModelParams below.  Deriving
# it means switching rasters in LAYER_PATHS needs no other edits.
N_SPATIAL_CHANNELS = sum(1 if arr.ndim == 2 else arr.shape[0] for arr in env_layers.values())
print(f"Spatial channels: {N_SPATIAL_CHANNELS}")

###  Plot each spatial covariate with the observed GPS locations overlaid.
This confirms the raster extent covers the animal's home range and gives a visual sense of the landscape the model will learn from.

In [ ]:
# Flatten env_layers into one (2-D array, label) pair per spatial channel so the
# figure adapts to whatever LAYER_PATHS provides (1 layer of 3 bands, 2 layers of
# 1 band, ...).  Band descriptions come from the GeoTIFF metadata where present.
plot_n_layers = 9  # limit the number of layers plotted to avoid a huge figure
channel_plots = []
for name, arr in env_layers.items():
    if arr.ndim == 2:
        channel_plots.append((arr, name))
        continue
    with rasterio.open(LAYER_PATHS[name]) as src:
        descs = src.descriptions
    for band in range(arr.shape[0]):
        label = descs[band] if descs and descs[band] else f"band {band + 1}"
        channel_plots.append((arr[band], f"{name}: {label}"))

channel_plots = channel_plots[:plot_n_layers]
ncol = min(3, len(channel_plots))
nrow = math.ceil(len(channel_plots) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 4.6 * nrow), squeeze=False)

for ax, (arr, label) in zip(axes.flat, channel_plots, strict=False):
    rasterio.plot.show(arr, transform=raster_transform, ax=ax, cmap="viridis")
    # Points rather than a connected line: raw_df holds 28 individuals, so a
    # single line would draw spurious jumps between them.
    ax.plot(
        raw_df["x1_"], raw_df["y1_"],
        "r.", alpha=0.3, markersize=0.7, label="GPS fixes",
    )
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    ax.set_title(label)
    ax.legend(loc="upper right", markerscale=10)

# Blank out any unused panels in the grid
for ax in axes.flat[len(channel_plots):]:
    ax.axis("off")

plt.suptitle("Spatial covariates with observed GPS locations", y=1.01)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "spatial_covariates.png"), dpi=100)
plt.show()

## `prepare_movement_df()`
Converts a fixes dataframe (one row per GPS location) into a *step* dataframe (one row per movement between consecutive fixes).

- For each step it computes:
    * `dx`, `dy`         — displacement in projected CRS units (metres)
    * `bearing`          — direction of travel (radians; 0 = north, clockwise positive)
    * `bearing_tm1`      — bearing of the *previous* step, used as a model input so the network can capture directional persistence by estimating a turning angle
    * `dt_hour`          — time elapsed between consecutive fixes (hours)
    * `hour_t1`, `yday_t1` — decimal hour-of-day and day-of-year at the step start location
    * `*_sin1` / `*_cos1`  — sine/cosine encodings of hour and yday (see SCALAR_COLS below)

The "1" on column names (x1_, y1_, t1_, etc.) flags them as "start of step" variables; and the "2" in x2_, y2_, t2_ are the corresponding "end of step" values, following the convention of the `amt` package [@Signer2019-fi]. The last fix for each individual is dropped because no forward step can be formed.

In [ ]:
# The CSV is already in amt step format, but prepare_movement_df() expects one row
# per *fix* and rebuilds the steps itself (it also derives bearing, dt_hour and the
# cyclic time encodings the model needs).  Point it at the step-start columns and
# let it re-form the steps from consecutive fixes.
step_df = deepssf.prepare_movement_df(
    raw_df, 
    id_col="name", 
    time_col="t1_", 
    x_col="x1_", 
    y_col="y1_"
)
n_dropped = len(raw_df) - len(step_df)
print(f"Movement steps : {len(step_df):,}  ({n_dropped} dropped — last per individual)")
step_df.head()

## `SCALAR_COLS`
These are non-spatial inputs. For processing with convolutional layers, they are converted into spatial windows (rasters/arrays), and stacked with the truly spatial covariates.

The model receives a fixed-size spatial window centred on each step's start location.  To let the network express time-varying habitat preferences (e.g. different nocturnal vs diurnal use), scalar covariates are tiled into additional channels of the same H × W size, so every pixel "knows" the current time context when habitat suitability is estimated.

- `hour_t1_sin1` / `hour_t1_cos1`
    * Sine and cosine of (2π × hour / 24), encoding time-of-day as a *circular* variable.  The circular encoding means 23:00 and 01:00 are treated as adjacent — a plain hour value would have an artificial discontinuity at midnight.

- `yday_t1_sin1` / `yday_t1_cos1`
    * Same circular encoding for day-of-year (1–365), capturing seasonal variation in habitat use (e.g. dry-season vs wet-season movements).

- `dt_hour`
    * Elapsed time between consecutive fixes (hours).  Because GPS collars do not always fix at a perfectly regular rate, including dt_hour lets the movement kernel rescale expected displacement proportionally to the actual time interval rather than assuming a fixed fix rate. Note: dt_hour must be supplied at *simulation* time as well, so that `simulate_trajectory()` produces inputs consistent with training.

In [ ]:
SCALAR_COLS = ["hour_t1_sin1",
               "hour_t1_cos1", 
               "yday_t1_sin1", 
               "yday_t1_cos1", 
               "dt_hour"]


## `PIXEL_SIZE`
Raster resolution in CRS units (metres here).  Read straight off `raster_transform`
rather than typed in: the transform is the authority on the grid the model is
trained against, and a hand-entered value that disagrees with it silently
mis-scales the movement kernel — which is parameterised in metres — against a
grid measured in pixels.

In [ ]:
# transform.a is the cell width, transform.e the (negative) cell height
PIXEL_SIZE = abs(raster_transform.a)

if not math.isclose(abs(raster_transform.a), abs(raster_transform.e)):
    raise ValueError(
        f"Non-square pixels ({abs(raster_transform.a)} x {abs(raster_transform.e)}); "
        "the model assumes square cells."
    )

print(f"Pixel size: {PIXEL_SIZE} m")

## `WINDOW_SIZE`
The size of the local availability window in cells/pixels.

The maximum step length that the model will allow is the distance between the centre cell 
and the nearest edge (in reality some steps may be longer if they are in the corners).

In [ ]:
WINDOW_SIZE = 51

print(f"Pixel size: {PIXEL_SIZE} m  →  window {WINDOW_SIZE} px = "
      f"{WINDOW_SIZE * PIXEL_SIZE:,.0f} m across")

print(f"Maximum possible step: {((WINDOW_SIZE - 1) / 2) * PIXEL_SIZE:,.0f} m")

## `BATCH_SIZE`
Number of training samples (movement steps) per training mini-batch

In [ ]:
BATCH_SIZE  = 128

## `N_CONV_LAYERS_MOVE`
How many convolutional layers in the movement subnetwork of the model?

Each of the convolution layers has a max pooling layer, so more layers will result in greater dimension reduction that will feed into the fully-connected layer that outputs the movement parameters.

In [ ]:
N_CONV_LAYERS_MOVE = 4

## `N_CONV_LAYERS_HAB`
How many convolutional layers in the habitat selection subnetwork of the model?

These convolution layers do not have max pooling layers, so there is no dimension reduction. More layers will increase the depth of the model and allow for more abstracted features to be identified, but will slow model training/prediction.

This count includes the final convolution layer, which is the one that collapses the feature maps down to the single-channel habitat selection surface, so the minimum is 1.

In [ ]:
N_CONV_LAYERS_HAB = 8

## `OUTPUT_CHANNELS`
Number of convolutional filters (and therefore feature maps) for **each** convolution layer. More convolutional filters can provide more flexible functions, but will increase the number of parameters and slow down training times.

In [ ]:
OUTPUT_CHANNELS = 8

### Pre-calculate the flattened CNN output size
This value will be the input to the fully-connected layers in the movement subnetwork, and is calculated here so that ModelParams can be set without hard-coding it.

`flattened_conv_dim()` reproduces exactly what the layers inside `ConvJointModel` do to the spatial dimensions:
* conv (padding=1) keeps the spatial dimension unchanged at each layer
* maxpool k=2, s=2 halves the spatial dimension (floor division) at each layer

Because every movement conv layer is paired with a max pooling layer, this size depends on `N_CONV_LAYERS_MOVE` — so it has to be recalculated whenever that (or `WINDOW_SIZE`, or `OUTPUT_CHANNELS`) changes. It raises an error if the window is pooled away to nothing, which means too many layers for the window size.

In [ ]:
DENSE_DIM = deepssf.flattened_conv_dim(
    image_dim=WINDOW_SIZE,
    n_conv_layers_move=N_CONV_LAYERS_MOVE,
    output_channels=OUTPUT_CHANNELS,
)
print(f"After {N_CONV_LAYERS_MOVE}x conv+maxpool: dense_dim_in_all={DENSE_DIM}")

## `filter_steps_by_window()` 
Removes steps whose straight-line displacement exceeds half the spatial window (WINDOW_SIZE × PIXEL_SIZE / 2 metres). Such steps would place the true next location outside the predicted probability map, causing an index error in the loss function.

In [ ]:
original_data_size = len(step_df)

step_df = deepssf.filter_steps_by_window(
    step_df, window_size=WINDOW_SIZE, pixel_size=PIXEL_SIZE
)

print(f"Initial number of steps: {original_data_size:,}")
print(f"Steps after window filter: {len(step_df):,}")
print(f"Data reduction: {original_data_size - len(step_df):,} ({(original_data_size - len(step_df)) / original_data_size * 100:.1f}%)")

## `make_dataloaders()`
Splits step_df into train / val / test subsets (here 80 / 10 / 10 %) and returns a PyTorch DataLoader for each.

**The split is a `torch.utils.data.random_split`, not a chronological one** — each subset is a random sample of steps drawn from across the whole dataset and all 28 individuals. Consecutive steps are highly autocorrelated, so a random split leaks information between train and test and will make the validation numbers below look better than genuine out-of-sample performance. If you want an honest held-out estimate, split `step_df` yourself (by individual, or by time) and build the DataLoaders from the resulting subsets.

- Each DataLoader yields tuples of:
    * `x1`
        — spatial window tensor  (batch × n_spatial_channels × H × W)
        - One channel per raster *band* in LAYER_PATHS, cropped around each step's start pixel.
    * `x2`
        — scalar covariate tensor (batch × len(SCALAR_COLS))
        - Raw scalar values; broadcast into spatial maps *inside* the model.
    * `x3`
        — previous bearing tensor (batch × 1)
        - Conditions the movement kernel on the direction of the prior step.
    * `(px2, py2)`
        — pixel indices of the *true* next location within the window
        - Used by the loss function to look up the predicted probability at the observed destination.

The layer_paths argument causes the function to re-read the rasters internally, so the DataLoader is independent of the env_layers dict loaded above.

In [ ]:
dl_train, dl_val, dl_test = deepssf.make_dataloaders(
    layer_paths=LAYER_PATHS,
    window_size=WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    train_split=0.8,
    val_split=0.1,
    scalar_cols=SCALAR_COLS,
    df=step_df,
)
print(f"Train batches : {len(dl_train)}")
print(f"Val   batches : {len(dl_val)}")
print(f"Test  batches : {len(dl_test)}")

### Inspect one batch to verify tensor shapes match ModelParams expectations
`x1` has one channel per spatial *band* (NDVI + NDWI + NDMI = 3 with the index raster).  The scalar covariates in `x2` are broadcast to full H × W maps *inside* ConvJointModel and concatenated with `x1` there, giving the CNN N_SPATIAL_CHANNELS + len(SCALAR_COLS) = 3 + 5 = 8 input channels total.

The `px2` and `py2` are the target that the model is trying to predict, and are the pixel values of the next step (with the centre cell being (50, 50))

*`next(iter(dl_train))` will call the next batch, which are randomly shuffled, so calling the function again will provide different training samples.*

In [ ]:
x1, x2, x3, (px2, py2), _ = next(iter(dl_train))
print(f"Spatial   x1 : {x1.shape}   (batch × channels × H × W)")
print(f"Scalars   x2 : {x2.shape}   (batch × len(SCALAR_COLS))")
print(f"Bearing   x3 : {x3.shape}   (batch × 1)")
print(f"Next-step px2: {px2[:4].tolist()}  py2: {py2[:4].tolist()}")

### Plot the first set of tiles in the batch

In [ ]:
# Plot the spatial window "tiles" from the first batch: one row per sample, one
# column per raster band.  The centre pixel is the step's start location and the
# cross marks the true next location that the loss function will look up, so this
# is the quickest way to confirm the crops and targets line up with the rasters.
n_samples = 4  # how many samples from the batch to display
n_channels = 6 # number of spatial channels (bands) in the model input
# n_channels = x1.shape[1] # to show all layers, or set to a smaller number to avoid a huge figure

# Band labels straight from the GeoTIFF metadata, in the channel order the model sees
channel_labels = []
for name, arr in env_layers.items():
    if arr.ndim == 2:
        channel_labels.append(name)
        continue
    with rasterio.open(LAYER_PATHS[name]) as src:
        descs = src.descriptions
    for band in range(arr.shape[0]):
        channel_labels.append(descs[band] if descs and descs[band] else f"{name} b{band + 1}")

n_show = min(n_samples, x1.shape[0])
n_chan = min(n_channels, x1.shape[1])
centre = WINDOW_SIZE // 2

fig, axes = plt.subplots(
    n_show, n_chan, figsize=(3.0 * n_chan, 3.0 * n_show), squeeze=False
)

for row in range(n_show):
    for col in range(n_chan):
        ax = axes[row, col]
        tile = x1[row, col].numpy()
        ax.imshow(tile, cmap="viridis", origin="upper")
        # Start location: always the centre cell of the window by construction
        ax.plot(centre, centre, "w+", markersize=9, markeredgewidth=1.5)
        # True next step — px2 is the column index, py2 the row index
        ax.plot(px2[row].item(), py2[row].item(), "rx", markersize=9, markeredgewidth=1.5)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(channel_labels[col], fontsize=10)
        if col == 0:
            ax.set_ylabel(
                f"sample {row}\n→ ({px2[row].item()}, {py2[row].item()})", fontsize=8
            )

fig.suptitle(
    f"First batch: {WINDOW_SIZE}×{WINDOW_SIZE} tiles  "
    "(white + = step start, red × = observed next step)",
    y=1.0,
)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "batch_tiles.png"), dpi=100, bbox_inches="tight")
plt.show()


## `ModelParams` 
A thin dict-like container (it is a wrapper around something else rather than storing data) that groups all architecture hyper-parameters so they can be passed to ConvJointModel and logged together.

This is where the parameters that are passed to the model are set.


In [ ]:
params = deepssf.ModelParams({
    "batch_size":                BATCH_SIZE,
    "image_dim":                 WINDOW_SIZE,           # spatial window size (pixels)
    "pixel_size":                PIXEL_SIZE,            # metres per pixel
    "dim_in_nonspatial_to_grid": len(SCALAR_COLS),      # scalar cols tiled into window
    "dense_dim_in_nonspatial":   len(SCALAR_COLS),      # scalar inputs to the dense MLP
    "dense_dim_hidden":          128,                   # hidden units in the scalar MLP
    "dense_dim_in_all":          DENSE_DIM,             # flattened CNN output size
    # One channel per raster band (not per layer) + the tiled scalar covariates
    "input_channels":            N_SPATIAL_CHANNELS + len(SCALAR_COLS),
    "output_channels":           OUTPUT_CHANNELS,       # feature maps per conv layer
    "kernel_size":               3,                     # conv kernel spatial size
    "stride":                    1,                     # conv stride (should be kept at 1)
    "kernel_size_mp":            2,                     # max-pool kernel size
    "stride_mp":                 2,                     # max-pool stride 
    "padding":                   1,                     # same-padding preserves spatial dimensions
    "num_movement_params":       12,                    # movement kernel parameters
    "n_conv_layers_hab":         N_CONV_LAYERS_HAB,     # conv layers, habitat branch
    "n_conv_layers_move":        N_CONV_LAYERS_MOVE,    # conv+maxpool layers, movement
    "dropout":                   0.0,
    "device":                    DEVICE,
})
print(f"input_channels: {params.input_channels}  (must match x1 channels + len(SCALAR_COLS))")

## `ConvJointModel` 
The deepSSF model! A joint habitat–movement model with two heads:
1. Habitat head 
    — a small CNN applied to the spatial window (env layers + tiled scalars) that outputs a per-pixel log-suitability map.
2. Movement head 
    — a dense MLP (fully-connected layer) that takes the scalar covariates and the previous bearing, and produces parameters for a parametric movement kernel (a mixture of wrapped distributions centred on the current location).

The two log-probability surfaces are summed pixel-wise to give the joint predicted probability map for the animal's next step.

In [ ]:
model = deepssf.ConvJointModel(params).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {n_params:,}")

### Print the model structure

In [ ]:
print(model)

## `negativeLogLikeLoss` 
Scores each predicted probability map against the true next-step pixel index; minimising it pushes probability mass toward observed locations.

In [ ]:
loss_fn = deepssf.negativeLogLikeLoss(reduction="mean")

## `make_optimisers()`
Creates two Adam optimisers with separate learning rates:

- `lr_habitat`
    * controls the CNN (habitat) head
- `lr_movement`
    * controls the movement branch (`conv_movement` + `fcn_movement_all`), kept lower for stability

It also returns a `ReduceLROnPlateau` scheduler for each optimiser that reduces the learning rate by a factor of `scheduler_factor` (default = 0.1) if that head's validation loss does not improve for `scheduler_patience` epochs (default = 5).

Note that each scheduler now watches **its own** component of the validation loss. Previously both watched the combined loss, which is dominated by the movement component — so the movement head's plateau was what cut the habitat learning rate. With that fixed, the two learning rates no longer have to be separated by two orders of magnitude to stop movement from running away with training; start closer together and tune from there.


In [ ]:
optimisers, schedulers = deepssf.make_optimisers(
    model, 
    lr_habitat=1e-4, 
    lr_movement=1e-5, 
    scheduler_patience=3, 
    scheduler_factor=0.5
)

## `EarlyStopping()`

Defines when the model should stop training.

- `patience`
    * If the loss doesn't improve for `patience` epochs, terminate training

In [ ]:
early_stop = deepssf.EarlyStopping(
    patience=10,
    verbose=True,
    monitor="both",                 # stop only when BOTH heads have plateaued
    path=str(OUTPUT_DIR / "best_model.pt"),
    checkpoint_on="active",         # a habitat-only stage saves on habitat loss
    # Also keep each head's own best epoch, across the whole run
    head_paths={
        "habitat":  str(OUTPUT_DIR / "best_model_habitat.pt"),
        "movement": str(OUTPUT_DIR / "best_model_movement.pt"),
    },
)

## `fit()`
Runs the full training loop.

- Inputs:
    * `model`
        - the name of the deepSSF model specified using `ConvJointModel` above
    * `image_trim_pixels`
        - how many outer pixel rings to trim from the output images before saving to prevent artifacts with the colour scaling due to the image padding being pulled into the image during the convolution layers.
    * `dl_train`
        - train dataloader (contains the training data)
    * `dl_val`
        - validation dataloader (contains the validation data used to assess the learning rate and early stopping)
    * `loss_fn`
        - loss function
    * `optimisers`
        - optimisers used to update the model weights
    * `schedulers`
        - learning-rate schedulers (used to update the learning rates)
    * `stages`
        - optional coordinate-ascent schedule; each stage names which sub-networks receive updates (see below)
    * `early_stopping`
        - stores the validation loss and stops the model training if it doesn't improve for a given number of epochs (set using `patience` in the `EarlyStopping` definition)
    * `snapshot_dir`
        - where to save images of the training progress
    * `snapshot_item`
        - which sample (movement step) from the validation dataset to run the model on to generate the movement, habitat selection and next-step probability surfaces

- Outputs:
    * history dict with keys `train_losses`, `val_losses`, `val_habitat_losses`, `val_movement_losses` and `stage`

**Index 0 of every history list is a pre-training baseline** — a validation pass run before the first epoch, with `train_losses[0] = nan` and `stage[0] = -1`. An untrained habitat surface is uniform over the window, so its baseline should land near `log(75 * 75) = 8.635`; the gap between that and the final `val_habitat_losses` is a direct read on how much the habitat head actually learned.

### Staged training

`stages` runs the two sub-networks as a coordinate ascent on the same joint likelihood, rather than reweighting it:

1. **movement only** — get a sensible availability kernel in place first, since habitat is fitted *conditional* on it
2. **habitat only** — habitat now runs to its own convergence, on its own early-stopping clock, with the movement surface held fixed in the likelihood as an offset
3. **both** — a joint pass to settle

A frozen sub-network keeps contributing its surface to the loss; only its weights stop moving. This is why staging is used instead of `negativeLogLikeLoss(freeze_movement=True)` — that flag *drops* the movement surface from the combined loss, so habitat would have to explain the movement structure too and would learn a distance-decay bump centred on the previous location, which is the opposite of a clean resource-selection surface.

Early stopping ends the current *stage* and advances to the next, so the final stage's early stop ends the run.

Every epoch, a plot of the training and validation loss, and the predicted probability maps (movement, habitat selection and next-step) are written to `SNAPSHOT_DIR`, which will be turned into a GIF in the next cell.


In [ ]:
history = deepssf.fit(
    model=model,
    image_trim_pixels=4,
    window_size=WINDOW_SIZE,
    dl_train=dl_train,
    dl_val=dl_val,
    loss_fn=loss_fn,
    optimisers=optimisers,
    schedulers=schedulers,
    stages=[
        {"epochs": 10, "train": ("movement",)},   # availability kernel first
        {"epochs": 40, "train": ("habitat",)},    # habitat to its own convergence
        {"epochs": 30, "train": ("habitat", "movement")},  # joint settle
    ],
    early_stopping=early_stop,
    snapshot_dir=str(SNAPSHOT_DIR),
    snapshot_item=250,
    batch_size=BATCH_SIZE,
)

## `create_gif()`
Assembles the snapshots in `snapshot_dir` into an animation so you can see how
the predictions evolve as training progresses.

Display it with `IPython.display.Image`, which reads the file and embeds the
bytes in the notebook's output.  A hand-written `<img src="outputs/....gif">`
tag does *not* work here: the notebook's HTML output is rendered in a sandboxed
webview whose base URL is not the notebook's directory, so a relative path
resolves to nothing and you get an empty box.  Nothing is generating a still
frame — there is simply no image being loaded at all.  Embedding also means the
animation survives being exported to HTML or shared as a standalone `.ipynb`.

In [ ]:
gif_path = str(OUTPUT_DIR / "training_progress.gif")
deepssf.create_gif(str(SNAPSHOT_DIR), gif_path, fps=5)
print(f"Training GIF → {gif_path}")

# Image() embeds the file's bytes in the output, so it renders wherever the
# notebook is opened.  Leave it as the last expression to display it.
Image(filename=gif_path)

## Training - loss across epochs

In [ ]:
# Three panels rather than one: the habitat and movement components move on very
# different scales, so sharing an axis would flatten the habitat curve into a
# horizontal line.  x = 0 is the pre-training baseline recorded by fit().
uniform_nll = np.log(WINDOW_SIZE ** 2)
epochs = np.arange(len(history["val_losses"]))
stages = np.asarray(history["stage"])

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)

panels = [
    ("Combined", history["val_losses"], history["train_losses"]),
    ("Habitat", history["val_habitat_losses"], None),
    ("Movement", history["val_movement_losses"], None),
]

for ax, (title, val, train) in zip(axes, panels):
    if train is not None:
        ax.plot(epochs, train, color="0.6", lw=1.5, label="train")
    ax.plot(epochs, val, color="C0", lw=1.5, label="val")
    if title == "Habitat":
        # A uniform surface over the window scores log(H*W), so this line is the
        # "learned nothing" mark — the whole question for the habitat head is how
        # far below it the curve gets.  Movement sits far below it from the first
        # epoch, and drawing it there would only compress that panel's axis.
        ax.axhline(uniform_nll, color="0.4", ls="--", lw=1,
                   label=f"uniform ({uniform_nll:.2f})")
    # Shade the staged-training blocks so the schedule is readable off the curve.
    for s in np.unique(stages[stages >= 0])[1::2]:
        within = epochs[stages == s]
        ax.axvspan(within.min() - 0.5, within.max() + 0.5, color="0.92", zorder=0)
    ax.set_xlabel("Epoch (0 = untrained baseline)")
    ax.set_ylabel("NLL (nats)")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Training history — combined loss and its two components")
plt.savefig(str(OUTPUT_DIR / "loss_history.png"), dpi=100)
plt.show()


## Choosing which checkpoint to use
`fit()` leaves the model holding its **final-epoch** weights, not the best ones,
so load a checkpoint before testing or simulating.

- `best_model.pt` — the model to use for **simulation**.  Both heads from one
  epoch, so the joint likelihood is the one that was actually measured.
- `best_model_habitat.pt` / `best_model_movement.pt` — each head at its own best
  epoch, tracked across the whole run.

Mixing them is a deliberate trade: taking the habitat head from a different
epoch gives the sharpest habitat surface to *map*, but the two heads then come
from different epochs and the joint model's likelihood is no longer what either
checkpoint recorded.  Use it for the landscape predictions below, not for
simulating trajectories.

In [ ]:
# The joint model — what the simulations further down should use
ckpt = deepssf.load_checkpoint(str(OUTPUT_DIR / "best_model.pt"), model)
print(f"best_model.pt         saved on '{ckpt['checkpoint_criterion']}' "
      f"= {ckpt['checkpoint_score']:.6f}")

hab_ckpt = deepssf.load_checkpoint(str(OUTPUT_DIR / "best_model_habitat.pt"))
print(f"best_model_habitat.pt  saved on '{hab_ckpt['checkpoint_criterion']}' "
      f"= {hab_ckpt['checkpoint_score']:.6f}")

# How much habitat was left on the table by the last epoch.  The two numbers
# above are different quantities (a combined loss and a habitat loss) and are
# not comparable directly, so read the gap off the history instead.
hab_hist = np.asarray(history["val_habitat_losses"])
best_epoch = int(np.nanargmin(hab_hist))
print(f"\nhabitat NLL: best {hab_hist[best_epoch]:.4f} at epoch {best_epoch}, "
      f"final {hab_hist[-1]:.4f} — a gap of {hab_hist[-1] - hab_hist[best_epoch]:+.4f} nats")
print(f"(a uniform surface over the window scores {np.log(WINDOW_SIZE ** 2):.4f} - "
      f"a gap of {hab_hist[-1] - np.log(WINDOW_SIZE ** 2):+.4f}.")

# To map the sharpest habitat surface instead, overwrite just that head:
# load_head_weights(str(OUTPUT_DIR / "best_model_habitat.pt"), model, "habitat")

# Test the model against withheld data

## Convert the NumPy rasters loaded earlier into PyTorch tensors.
Both validate_next_step_probs() and simulate_trajectory() call `get_landscape(month_index)` at each step to fetch the current environmental layers as a list of 2-D tensors.

With static (non-seasonal) rasters the month_index argument is unused. For seasonal data (e.g. monthly NDVI composites) you would index into a stack of layers here based on the current month.

In [ ]:
# validate_next_step_probs() and simulate_trajectory() both stack the list returned
# by get_landscape() along a new channel dimension, so every element must be a *2-D*
# tensor.  Unbind the band dimension of any multi-band layer so the channel order
# matches what MovementDataset fed the model during training.
landscape_tensors = []
for arr in env_layers.values():
    t = torch.from_numpy(np.asarray(arr).astype(np.float32))
    landscape_tensors.extend([t] if t.ndim == 2 else list(t))

print(f"Landscape channels: {len(landscape_tensors)}  (each {tuple(landscape_tensors[0].shape)})")

def get_landscape(_month_index):
    # Returns one 2-D tensor per spatial channel, in the same order as LAYER_PATHS.
    return landscape_tensors

# Take the last 10 % of steps as a test sample.  CAVEAT: make_dataloaders() uses a
# random_split, so these steps were mostly *seen during training* — the numbers below
# are not a clean out-of-sample estimate.  Because step_df is ordered by individual
# name, this slice is also just the last few pigs alphabetically rather than the
# latest steps in time.  Hold out individuals or a date range explicitly for a real test.
n_test      = int(len(step_df) * 0.1)
test_sample = step_df.iloc[-n_test:].reset_index(drop=True)
print(f"Testing on {len(test_sample):,} steps")

## `validate_next_step_probs()`
Runs the model forward on each test step and records the probability mass assigned to the *true* next location.

- **Inputs**: 
    * `model`
    * `test_sample`
        - test dataframe that contains movement steps (from the data loaded above)
     * `get_landscape`
        - retrieves the landscape layers (spatial covariates) that were used to train the model
    * `transform`
        - raster Affine transform
    * `window_size`
        - window size of the local availability window (should be the same as for training)
    * `scalar_cols`
        - scalar column names (should be set from above)
    * `yday_col`
        - which column has the yday column to ensure the correct spatial layers (if dynamic through time) are indexed
    * `month_index_fn`
        - a function mapping day-of-year → month index (used by the `get_landscape` function). Here we are using static spatial layers so we can set it to 0.

- **Outputs**: 
    * `test_sample` dataframe with three new columns appended:
        - habitat_prob   — probability at the true location from the habitat head alone
        - move_prob      — probability at the true location from the movement head alone
        - next_step_prob — joint (habitat × movement) probability at the true location

Higher values mean the model performed better (there was higher predicted probability at the observed next-step). 


In [ ]:
# Use the last 10 % of steps as the test set (matches the DataLoader split)
n_test      = int(len(step_df) * 0.1)
test_sample = step_df.iloc[-n_test:].reset_index(drop=True)
print(f"Validating on {len(test_sample):,} steps")

val_results = deepssf.validate_next_step_probs(
    model,
    test_sample,
    get_landscape=get_landscape,
    transform=raster_transform,
    window_size=WINDOW_SIZE,
    scalar_cols=tuple(SCALAR_COLS),
    yday_col="yday_t1",
    bearing_col="bearing_tm1",
    month_index_fn=lambda _yday: 0,   # static layers — month doesn't matter
)
print(val_results[["habitat_prob", "move_prob", "next_step_prob"]].describe())

## Plot the predictive performance on the held-out data

In [ ]:
# The uniform probability of any cell in the window is 1/(W*H) = 1/(W^2) since W=H.
uniform_prob = 1/(WINDOW_SIZE ** 2)
uniform_prob

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col, title in zip(
    axes,
    ["habitat_prob", "move_prob", "next_step_prob"],
    ["Habitat", "Movement", "Next-step"],
    strict=True,
):
    data = val_results[col].dropna()
    ax.hist(data, bins=40, edgecolor="white")
    ax.axvline(
        uniform_prob,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label="Uniform (1/W²)",
    )
    ax.set_title(title)
    ax.set_xlabel("Probability")
    ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, 3))
    ax.legend()

axes[0].set_ylabel("Count")
plt.suptitle("Next-step probability distributions (test set)", y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "validation_probs.png"), dpi=100, bbox_inches="tight")
plt.show()